# 03 — LSTM RUL Prediction

Train a **stacked LSTM with Bahdanau attention** on NASA CMAPSS FD001 to predict
Remaining Useful Life (RUL) of turbofan engines.

Topics covered:
1. Load and preprocess CMAPSS data
2. Build and train the LSTM model
3. Evaluate with RMSE, MAE, and NASA asymmetric score
4. Visualise predictions and attention weights
5. Export to ONNX for deployment

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Data Loading

In [ ]:
from datasets.cmapss_loader import CMAPSSLoader

loader = CMAPSSLoader(
    subset='FD001',
    data_dir='../datasets/data',
    max_rul=125,
    window_size=30,
    stride=1,
    normalize=True,
)

try:
    X_train, y_train, X_test, y_test = loader.load()
    data_loaded = True
except FileNotFoundError:
    print('CMAPSS data not found — using synthetic data for demonstration.')
    np.random.seed(42)
    X_train = np.random.randn(8000, 30, 14).astype(np.float32)
    y_train = np.random.uniform(0, 125, 8000).astype(np.float32)
    X_test  = np.random.randn(100, 30, 14).astype(np.float32)
    y_test  = np.random.uniform(0, 125, 100).astype(np.float32)
    data_loaded = False

# Train / validation split
np.random.seed(0)
n_val = int(len(X_train) * 0.1)
idx = np.random.permutation(len(X_train))
X_tr, y_tr = X_train[idx[n_val:]], y_train[idx[n_val:]]
X_val, y_val = X_train[idx[:n_val]], y_train[idx[:n_val]]

print(f'Train:      {X_tr.shape}')
print(f'Validation: {X_val.shape}')
print(f'Test:       {X_test.shape}')

## 2. Model Architecture

In [ ]:
from models.lstm_predictive import LSTMPredictiveModel

config = {
    'input_size':   X_tr.shape[-1],   # 14 sensor features
    'hidden_size':  128,
    'num_layers':   2,
    'dropout':      0.2,
    'lr':           1e-3,
    'weight_decay': 1e-5,
    'batch_size':   256,
    'epochs':       50,
    'patience':     10,
}

model = LSTMPredictiveModel(config)

n_params = sum(p.numel() for p in model.network.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')
print(model.network)

## 3. Training

In [ ]:
history = model.fit(X_tr, y_tr, X_val, y_val)

# Plot training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train_loss'], label='Train MSE loss', color='steelblue')
if 'val_loss' in history and history['val_loss']:
    ax.plot(history['val_loss'], label='Validation MSE loss', color='coral')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM Training Curves')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print(f'Best validation loss: {min(history.get("val_loss", [float("inf")])):.4f}')

## 4. Evaluation

In [ ]:
from evaluation.rul_metrics import evaluate_rul, plot_rul_predictions

y_pred = model.predict(X_test)
metrics = evaluate_rul(y_test, y_pred)

fig = plot_rul_predictions(y_test, y_pred, title='LSTM — RUL Prediction (CMAPSS FD001)')
plt.show()

## 5. Attention Weight Visualisation

In [ ]:
# Visualise what timesteps the model attends to
# Use a sample from the test set
sample_idx = np.argmin(y_test)   # unit closest to failure
X_sample = X_test[sample_idx: sample_idx + 1]   # (1, 30, 14)

pred_rul, attn_weights = model.predict_with_attention(X_sample)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Raw sensor signals for this window
axes[0].imshow(
    X_sample[0].T,
    aspect='auto', cmap='RdYlGn',
    interpolation='nearest'
)
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Sensor channel')
axes[0].set_title(f'Sensor values — Unit {sample_idx}  (True RUL={y_test[sample_idx]:.0f}, Predicted={pred_rul[0]:.1f})')
plt.colorbar(axes[0].images[0], ax=axes[0])

# Attention weights over time
axes[1].bar(range(30), attn_weights[0], color='steelblue', alpha=0.85)
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Attention weight')
axes[1].set_title('Bahdanau Attention Weights — most recent steps attended most')

plt.tight_layout()
plt.show()

## 6. Save Model & Export to ONNX

In [ ]:
import os
os.makedirs('../checkpoints', exist_ok=True)

# Save PyTorch checkpoint
model.save('../checkpoints/lstm_cmapss_fd001.pt')

# Export to ONNX (requires onnx package)
try:
    from deployment.edge_inference import export_to_onnx
    onnx_path = export_to_onnx(
        model,
        '../checkpoints/lstm_cmapss_fd001.onnx',
        input_shape=(1, 30, X_tr.shape[-1])
    )
    print(f'ONNX model saved: {onnx_path}')
except ImportError as e:
    print(f'ONNX export skipped: {e}')

## Summary

| Metric | Value |
|--------|-------|
| RMSE | see output above |
| MAE | see output above |
| NASA Score | see output above |

The Bahdanau attention mechanism helps the model focus on the **most recent timesteps**,
which carry the most degradation-relevant information.

**Next:** [04 — Transformer RUL](04_transformer_rul.ipynb)